## 1. 파일 경로 설정 및 라이브러리 임포트

In [10]:
"""
===================================================================================
1. 파일 경로 설정 및 라이브러리 임포트
===================================================================================
목적: 교통온도 계산에 필요한 라이브러리와 데이터 파일 경로를 설정합니다.

주요 라이브러리:
- pandas: 데이터프레임 처리
- numpy: 수치 연산
- json: JSON 파일 읽기
- requests: 카카오 지오코딩 API 호출
- haversine 관련 함수: 위도/경도로 거리 계산
===================================================================================
"""

import pandas as pd
import numpy as np
import json
import requests
import time
from math import radians, sin, cos, asin, sqrt
from pathlib import Path

# 파일 경로 설정
BASE_DIR = Path(r"c:\Users\Playdata\OneDrive\바탕 화면\ondo")

TRAFFIC_DIR = BASE_DIR / "ondo_traffic" / "data" / "교통"
LAND_DIR = BASE_DIR / "land_data" / "land"
SUBWAY_CSV = TRAFFIC_DIR / "서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv"
BUS_CSV = TRAFFIC_DIR / "국토교통부_전국 버스정류장 위치정보_20241031_utf8.csv"
STATION_COORDS_CSV = TRAFFIC_DIR / "지하철_노선도.csv"

LISTING_FILES = [
    LAND_DIR / "00_통합_원투룸_2.json",  # 현재는 원투룸만 사용
    # LAND_DIR / "00_통합_오피스텔.json",
    # LAND_DIR / "00_통합_빌라주택.json",
    # LAND_DIR / "00_통합_아파트.json",
    # LAND_DIR / "00_통합_상가.json",
]

print("데이터 파일 경로 설정 완료")
print(f"교통 데이터 폴더: {TRAFFIC_DIR}")
print(f"부동산 데이터 폴더: {LAND_DIR}")

데이터 파일 경로 설정 완료
교통 데이터 폴더: c:\Users\Playdata\OneDrive\바탕 화면\ondo\ondo_traffic\data\교통
부동산 데이터 폴더: c:\Users\Playdata\OneDrive\바탕 화면\ondo\land_data\land


## 2. 지하철 데이터 로드

In [11]:
"""
===================================================================================
2. 지하철 승하차 데이터 로드 및 처리
===================================================================================
목적: 
1. 지하철역별 승하차 인원 데이터를 로드(2015.01 ~ 2025.10)
2. 출퇴근 시간대(07-09시, 18-20시) 승하차 인원 집계
3. 역별로 출퇴근 비율과 총 승하차량 계산

핵심 지표:
- 일일_총승하차: 하루 전체 승하차 인원 (역 규모 파악)
- 출퇴근_승하차: 출퇴근 시간대 승하차 인원
- 출퇴근비율: 출퇴근 시간대 비중 (직장인 밀집도 파악)
===================================================================================
"""

print("=" * 60)
print("1. 지하철 승하차 데이터 처리")
print("=" * 60)

# CSV 파일 로드 (인코딩: cp949 = 한글 윈도우 인코딩)
subway = pd.read_csv(SUBWAY_CSV, encoding="cp949")
print(f"지하철 데이터 shape: {subway.shape}") 
print(f"지하철 데이터 컬럼: {subway.columns.tolist()[:10]}...") 

# 시간대별 승차/하차 컬럼 찾기
time_cols = [c for c in subway.columns if "승차인원" in c or "하차인원" in c]
print(f"\n시간대 컬럼 수: {len(time_cols)}")  # 48개 (24시간 × 2(승차+하차))

# 출퇴근 시간대 컬럼만 필터링
# 출근: 07-08시, 08-09시
# 퇴근: 18-19시, 19-20시
commute_cols = [
    c for c in time_cols
    if "07시-08시" in c or "08시-09시" in c
    or "18시-19시" in c or "19시-20시" in c
]
print(f"출퇴근 시간대 컬럼 수: {len(commute_cols)}")  # 8개 (4시간대 × 2(승차+하차))

# 핵심 지표 계산
# 1) 일일 총 승하차: 모든 시간대 승하차 인원 합계
subway["일일_총승하차"] = subway[time_cols].sum(axis=1)

# 2) 출퇴근 시간대 승하차: 출퇴근 시간대만 합계
subway["출퇴근_승하차"] = subway[commute_cols].sum(axis=1)

# 3) 출퇴근 비율: 출퇴근 승하차 / 일일 총 승하차
#    - 0으로 나누기 방지를 위해 replace(0, np.nan) 사용
subway["출퇴근비율"] = subway["출퇴근_승하차"] / subway["일일_총승하차"].replace(0, np.nan)

# 역(호선+역명) 단위로 월별 데이터를 평균 집계
# 예: "1호선-서울역"의 여러 달 데이터를 평균내어 하나의 행으로 만듦
station = subway.groupby(["호선명", "지하철역"]).agg(
    일일_총승하차=("일일_총승하차", "mean"),    # 평균 일일 승하차량
    출퇴근_승하차=("출퇴근_승하차", "mean"),    # 평균 출퇴근 승하차량
    출퇴근비율=("출퇴근비율", "mean"),          # 평균 출퇴근 비율
).reset_index()

# 결측치(NaN) 처리: 출퇴근비율이 NaN이면 0.0으로 대체
station["출퇴근비율"] = station["출퇴근비율"].fillna(0.0)

print(f"\n역 단위 집계 결과: {station.shape}")  # (713, 5) = 713개 역
station.head()  # 처음 5개 행 출력

1. 지하철 승하차 데이터 처리
지하철 데이터 shape: (77386, 52)
지하철 데이터 컬럼: ['사용월', '호선명', '지하철역', '04시-05시 승차인원', '04시-05시 하차인원', '05시-06시 승차인원', '05시-06시 하차인원', '06시-07시 승차인원', '06시-07시 하차인원', '07시-08시 승차인원']...

시간대 컬럼 수: 48
출퇴근 시간대 컬럼 수: 8

역 단위 집계 결과: (713, 5)


,호선명,지하철역,일일_총승하차,출퇴근_승하차,출퇴근비율
0,1호선,동대문,8.209110e+05,180491.523077,0.220075
1,1호선,동묘앞,6.036072e+05,110204.692308,0.182145
2,1호선,서울역,3.096310e+06,960684.300000,0.313200
3,1호선,시청,1.407915e+06,520211.584615,0.373452
4,1호선,신설동,8.548117e+05,277636.776923,0.325186


In [12]:
# 출퇴근 비율이 높은 역은 내림차순 정렬하여 상위 5개 선택
top5 = station.sort_values("출퇴근비율", ascending=False).head(5)
top5

,호선명,지하철역,일일_총승하차,출퇴근_승하차,출퇴근비율
617,수인선,남동인더스파크,1.206622e+05,6.054471e+04,0.501455
557,공항철도 1호선,공항화물청사,1.367660e+05,6.653642e+04,0.484820
265,7호선,가산디지털단지,2.298484e+06,1.058740e+06,0.460591
167,5호선,동대문역사문화공원,1.934020e+05,8.799660e+04,0.455799
192,5호선,여의도,1.470797e+06,6.628094e+05,0.452798


## 3. 지하철 좌표 데이터 로드

In [13]:
"""
===================================================================================
3. 지하철역 좌표 데이터 병합
===================================================================================
목적:
1. 지하철역의 위도/경도 정보를 로드
2. 승하차 데이터와 좌표 데이터를 병합
3. 역별 점수 계산을 위한 정규화 수행

정규화 방법 (Min-Max Normalization):
- 값을 0~1 범위로 변환
- 공식: (값 - 최소값) / (최대값 - 최소값)
===================================================================================
"""

print("\n" + "=" * 60)
print("2. 지하철역 좌표 데이터 병합")
print("=" * 60)

# Min-Max 정규화 함수 정의
def minmax_norm(s: pd.Series) -> pd.Series:
    """
    시리즈를 0~1 범위로 정규화
    
    Args:
        s: pandas Series (정규화할 데이터)
    
    Returns:
        정규화된 Series (0~1 사이 값)
    """
    s_min, s_max = s.min(), s.max()
    
    # 최대값과 최소값이 같으면 0으로 반환 (분산이 없는 경우)
    if s_max - s_min == 0:
        return pd.Series(0.0, index=s.index)
    
    # Min-Max 정규화 공식 적용
    return (s - s_min) / (s_max - s_min)

# 역별 점수 정규화 (0~1 범위)
# Peak_subway: 출퇴근 비율 정규화 (출퇴근 집중도)
station["Peak_subway"] = minmax_norm(station["출퇴근비율"])

# Size_subway: 일일 총 승하차 정규화 (역 규모)
station["Size_subway"] = minmax_norm(station["일일_총승하차"])

# 지하철역 좌표 데이터 로드
station_coords = pd.read_csv(STATION_COORDS_CSV, encoding="utf-8")
print(f"역 좌표 데이터 shape: {station_coords.shape}")  # (1090, 15)
print(f"역 좌표 데이터 컬럼: {station_coords.columns.tolist()}")

# 컬럼명 통일 (데이터셋 간 컬럼명이 다를 수 있음)
station_coords_clean = station_coords.rename(columns={
    "역사명": "지하철역",      # 역 이름
    "노선명": "호선명",        # 호선 이름 (1호선, 2호선 등)
    "역위도": "역_lat",       # 위도
    "역경도": "역_lng"        # 경도
})

# 병합에 필요한 컬럼만 선택
station_coords_clean = station_coords_clean[["호선명", "지하철역", "역_lat", "역_lng"]].copy()

# 승하차 데이터와 좌표 데이터 병합
# inner join: 양쪽 데이터에 모두 존재하는 역만 포함
station_full = station.merge(
    station_coords_clean,
    on=["호선명", "지하철역"],  # 호선명 + 역명으로 매칭
    how="inner"                 # 교집합만 사용
)

print(f"\n병합 후 역 데이터 shape: {station_full.shape}")  # (315, 9)
# 주의: 713개 역 중 315개만 좌표 데이터가 있음

station_full.head()  # 결과 확인


2. 지하철역 좌표 데이터 병합
역 좌표 데이터 shape: (1090, 15)
역 좌표 데이터 컬럼: ['역번호', '역사명', '노선번호', '노선명', '영문역사명', '한자역사명', '환승역구분', '환승노선번호', '환승노선명', '역위도', '역경도', '운영기관명', '역사도로명주소', '역사전화번호', '데이터기준일자']

병합 후 역 데이터 shape: (315, 9)


,호선명,지하철역,일일_총승하차,출퇴근_승하차,출퇴근비율,Peak_subway,Size_subway,역_lat,역_lng
0,1호선,동대문,8.209110e+05,180491.523077,0.220075,0.438178,0.158011,37.571646,127.010690
1,1호선,동묘앞,6.036072e+05,110204.692308,0.182145,0.362443,0.116184,37.573265,127.016459
2,1호선,서울역,3.096310e+06,960684.300000,0.313200,0.624117,0.595987,37.555850,126.972103
3,1호선,시청,1.407915e+06,520211.584615,0.373452,0.744421,0.270999,37.565211,126.977207
4,1호선,신설동,8.548117e+05,277636.776923,0.325186,0.648049,0.164536,37.576117,127.024710


## 4. 거리 계산 함수 및 역 점수 계산

In [14]:
"""
===================================================================================
4. 거리 계산 함수 및 역 점수 계산
===================================================================================
목적:
1. 위도/경도를 사용한 실제 거리 계산 함수 구현 (Haversine 공식)
2. 역별 기본 점수 계산 (거리 제외)

Haversine 공식:
- 구면상의 두 점 사이 최단 거리를 계산하는 공식
- 지구를 완전한 구로 가정하여 계산
- 정확도: 수 미터 오차 범위 (일반적인 용도로 충분함)

“지구 반지름(R)을 가진 공 위에 두 점이 있을 때 그 둘 사이의 **최단 거리(대권거리, great-circle distance)**를 구하자”
===================================================================================
"""

print("\n" + "=" * 60)
print("3. 역 점수 계산")
print("=" * 60)

def haversine(lat1, lon1, lat2, lon2):
    """
    Haversine 공식을 사용한 두 지점 간 거리 계산
    
    Args:
        lat1, lon1: 첫 번째 지점의 위도, 경도 (도 단위)
        lat2, lon2: 두 번째 지점의 위도, 경도 (도 단위)
    
    Returns:
        거리 (미터 단위)
    
    공식 설명:
    1. 위도/경도를 라디안으로 변환 (1도 = π/180 라디안)
    2. Haversine 공식으로 중심각 계산
    3. 중심각 × 지구 반지름 = 거리
    """
    R = 6371000  # 지구 반지름 (미터) - 평균값 사용
    
    # 도(degree)를 라디안(radian)으로 변환
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # 위도/경도 차이 계산
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine 공식 적용
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))  # 중심각
    
    return R * c  # 거리 = 반지름 × 중심각

# 역 단위 기본 점수 계산 (거리 점수는 매물마다 다르므로 나중에 계산)
# S_subway_station = 0.4 × Peak_subway + 0.2 × Size_subway
#
# 가중치 설명:
# - Peak_subway (출퇴근 비율): 0.4 가중치 → 출퇴근 집중도 중시
# - Size_subway (역 규모): 0.2 가중치 → 전체 이용객 수 반영
# - 거리 점수: 매물별로 다르므로 여기서는 제외 (나중에 추가)
station_full["S_subway_station"] = (
    0.4 * station_full["Peak_subway"]    # 출퇴근 비율 기여도
    + 0.2 * station_full["Size_subway"]  # 역 규모 기여도
)

# numpy 배열로 변환 (벡터화 연산으로 속도 최적화)
# 매물 1개당 315개 역과의 거리를 계산해야 하므로 속도가 중요함
station_coords_arr = station_full[["역_lat", "역_lng"]].to_numpy()
station_scores_arr = station_full["S_subway_station"].to_numpy()

print(f"역 좌표 배열 shape: {station_coords_arr.shape}")  # (315, 2)
print(f"역 기본 점수 통계:")
print(station_full["S_subway_station"].describe())
# count: 315개 역
# mean: 평균 0.30 (0~1 범위에서 중간 수준)
# std: 표준편차 0.044 (역 간 점수 차이가 크지 않음)
# min~max: 0.168 ~ 0.456 범위


3. 역 점수 계산
역 좌표 배열 shape: (315, 2)
역 기본 점수 통계:
count    315.000000
mean       0.300767
std        0.043711
min        0.168214
25%        0.273373
50%        0.299321
75%        0.321615
max        0.455847
Name: S_subway_station, dtype: float64


## 5. 부동산 매물 데이터 로드 및 지오코딩

In [15]:
"""
===================================================================================
5. 부동산 매물 데이터 로드
===================================================================================
목적:
1. JSON 형식의 부동산 매물 데이터 로드
2. 매물번호, 주소, 매물유형 정보 추출
3. 지오코딩 준비 (주소 → 위도/경도 변환 필요)

데이터 구조:
- 매물번호: 고유 식별자
- 주소: 카카오 API로 지오코딩할 텍스트
- property_type: 원투룸, 오피스텔, 빌라 등
===================================================================================
"""

print("\n" + "=" * 60)
print("4. 부동산 매물 데이터 로드")
print("=" * 60)

# 카카오 지오코딩 함수 정의 (주소 → 위도/경도 변환)
def geocode_address(address, kakao_api_key="2b552ce8cd4023635129f04f72cfd7f7"):
    """
    카카오 REST API를 사용해서 주소를 위도/경도로 변환
    
    Args:
        address: 검색할 주소 (예: "서울특별시 서초구 양재동 116-3")
        kakao_api_key: 카카오 개발자 앱의 REST API 키
    
    Returns:
        (위도, 경도) 튜플, 또는 실패 시 (None, None)
    """
    # API 키가 없으면 지오코딩 불가
    if not kakao_api_key:
        return None, None
    
    # 카카오 로컬 API 엔드포인트
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    
    # API 인증 헤더 (REST API 키 사용)
    headers = {"Authorization": f"KakaoAK {kakao_api_key}"}
    
    # 검색 파라미터
    params = {"query": address}
    
    try:
        # GET 요청 전송
        response = requests.get(url, headers=headers, params=params)
        
        # 성공 응답 확인 (HTTP 200)
        if response.status_code == 200:
            result = response.json()
            
            # 검색 결과가 있으면 첫 번째 결과 사용
            if result['documents']:
                doc = result['documents'][0]
                # y = 위도(latitude), x = 경도(longitude)
                return float(doc['y']), float(doc['x'])
    except Exception as e:
        # 에러 발생 시 무시하고 None 반환
        pass
    
    return None, None

# 부동산 매물 데이터 로드
all_listings = []

for listing_file in LISTING_FILES:
    # 파일이 존재하는지 확인
    if listing_file.exists():
        print(f"\n로드 중: {listing_file.name}")
        
        # JSON 파일 읽기
        with open(listing_file, encoding="utf-8") as f:
            data = json.load(f)
        
        # 각 매물 정보 추출
        for item in data:
            # 매물번호 추출
            listing_id = item.get("매물번호", "")
            
            # 주소 정보 추출
            address_info = item.get("주소_정보", {})
            address = address_info.get("전체주소", "")
            
            # 매물번호와 주소가 모두 있는 경우만 포함
            if listing_id and address:
                all_listings.append({
                    "listing_id": listing_id,
                    "address": address,
                    "property_type": listing_file.stem.replace("00_통합_", ""),  # 파일명에서 유형 추출
                    "lat": None,  # 지오코딩 후 채워질 예정
                    "lng": None,  # 지오코딩 후 채워질 예정
                })
        
        # 이 파일에서 로드한 매물 수 출력
        print(f"  로드된 매물 수: {len([x for x in all_listings if x['property_type'] == listing_file.stem.replace('00_통합_', '')])}")

# 리스트를 DataFrame으로 변환
listings = pd.DataFrame(all_listings)
print(f"\n총 매물 수: {len(listings)}")
print(listings.head())  # 처음 5개 매물 확인


4. 부동산 매물 데이터 로드

로드 중: 00_통합_원투룸_2.json
  로드된 매물 수: 6

총 매물 수: 6
  listing_id                        address property_type   lat   lng
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워         원투룸_2  None  None
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운         원투룸_2  None  None
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈         원투룸_2  None  None
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌         원투룸_2  None  None
4   18306808            서울특별시 서초구 양재동 366-9         원투룸_2  None  None


In [16]:
"""
===================================================================================
6. 지오코딩 수행
===================================================================================
목적:
1. 카카오 API를 사용해서 주소를 위도/경도로 변환
2. API 호출 제한을 고려한 속도 조절 (0.1초 대기)
3. 지오코딩 결과를 CSV 파일로 저장

주의사항:
- 카카오 API는 무료 플랜에서 일일 호출 제한이 있음
- 대량 데이터 처리 시 API 제한 확인 필요
- 현재 예제는 6개 매물로 테스트
===================================================================================
"""

print("\n" + "=" * 60)
print("5. 지오코딩 수행")
print("=" * 60)

# 카카오 API 키 설정
KAKAO_API_KEY = "2b552ce8cd4023635129f04f72cfd7f7"

if KAKAO_API_KEY:
    print("카카오 API를 사용해서 지오코딩을 수행합니다...")
    print(f"총 {len(listings)}개 주소 처리 예정")
    
    # 각 매물의 주소를 지오코딩
    for idx, row in listings.iterrows():
        # 진행 상황 출력 (100개마다)
        if idx % 100 == 0:
            print(f"  진행: {idx}/{len(listings)}")
        
        # 카카오 API 호출하여 위도/경도 받기
        lat, lng = geocode_address(row['address'], KAKAO_API_KEY)
        
        # DataFrame에 저장
        listings.at[idx, 'lat'] = lat
        listings.at[idx, 'lng'] = lng
        
        # API 호출 제한 방지: 0.1초 대기
        # (1초당 최대 10개 요청으로 제한)
        time.sleep(0.1)
    
    # 지오코딩 성공률 확인
    # lat과 lng가 모두 있는 행의 개수 계산
    success_count = listings[['lat', 'lng']].notna().all(axis=1).sum()
    print(f"\n지오코딩 성공: {success_count}/{len(listings)} ({success_count/len(listings)*100:.1f}%)")
    
    # 지오코딩 결과를 CSV 파일로 저장
    # encoding="utf-8-sig": 한글이 깨지지 않도록 BOM 포함 UTF-8 사용
    listings.to_csv(BASE_DIR / "listings_with_coords.csv", index=False, encoding="utf-8-sig")
    print("지오코딩 결과를 'listings_with_coords.csv'에 저장했습니다.")
    
else:
    print("카카오 API 키가 설정되지 않았습니다.")

# 유효한 좌표가 있는 매물만 필터링
# lat과 lng가 모두 NaN이 아닌 행만 선택
listings_valid = listings[listings[['lat', 'lng']].notna().all(axis=1)].copy()
print(f"\n유효한 좌표가 있는 매물: {len(listings_valid)}개")

# 결과 확인
if len(listings_valid) > 0:
    print(listings_valid.head())


5. 지오코딩 수행
카카오 API를 사용해서 지오코딩을 수행합니다...
총 6개 주소 처리 예정
  진행: 0/6

지오코딩 성공: 6/6 (100.0%)
지오코딩 결과를 'listings_with_coords.csv'에 저장했습니다.

유효한 좌표가 있는 매물: 6개
  listing_id                        address property_type        lat  \
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워         원투룸_2  37.474678   
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운         원투룸_2  37.465608   
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈         원투룸_2  37.474999   
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌         원투룸_2   37.46933   
4   18306808            서울특별시 서초구 양재동 366-9         원투룸_2  37.468881   

          lng  
0  127.035655  
1  127.034504  
2  127.041886  
3  127.044146  
4  127.045828  


## 6. 지하철 점수 계산

In [17]:
"""
===================================================================================
7. 지하철 점수 계산
===================================================================================
목적:
*역세권: 500m 이내 지하철역이 있는 지역
1. 역세권 거리에 따른 점수 함수 구현
2. 매물별로 가까운 역들을 찾아 종합 점수 계산

점수 체계:
- 1차 역세권 (0~250m): 점수 1.0 (만점)
- 2차 역세권 (250~500m): 점수 1.0→0.5 선형 감소
- 역세권 밖 (500m~): 점수 지수 감소 (0.5 × e^(-(d-500)/500))

가중치:
- 가까운 역 3개만 사용 (top_n=3)
- 거리 가중치: 1/(1 + d/250) → 가까운 역에 더 높은 가중치
===================================================================================
"""

print("\n" + "=" * 60)
print("6. 지하철 점수 계산 함수")
print("=" * 60)

def subway_distance_score(d):
    """
    역세권 거리에 따른 점수 계산
    
    Args:
        d: 역까지의 거리 (미터)
    
    Returns:
        거리 점수 (0~1 범위)
    
    점수 구간:
    - d ≤ 250m: 1.0 (1차 역세권, 도보 3분)
    - 250m < d ≤ 500m: 1.0 → 0.5 선형 감소 (2차 역세권, 도보 6분)
    - d > 500m: 0.5 × exp(-(d-500)/500) (역세권 밖, 점수 급감)
    
    예시:
    - 100m: 1.0 (만점)
    - 250m: 1.0 (경계)
    - 375m: 0.75 (중간)
    - 500m: 0.5 (경계)
    - 1000m: 0.18 (낮음)
    - 2000m: 0.01 (거의 0)
    """
    if d <= 250:
        # 1차 역세권: 만점
        return 1.0
    elif d <= 500:
        # 2차 역세권: 선형 감소
        # d=250일 때 1.0, d=500일 때 0.5
        return 1.0 - 0.5 * (d - 250) / 250
    else:
        # 역세권 밖: 지수 감소
        # 거리가 500m 증가할 때마다 점수가 e배 감소
        return 0.5 * np.exp(-(d - 500) / 500)

def compute_subway_score_for_property(p_lat, p_lng, top_n=3, max_dist=2000):
    """
    매물 1건에 대한 지하철 점수 계산
    
    Args:
        p_lat, p_lng: 매물의 위도, 경도
        top_n: 가장 가까운 상위 N개 역만 사용 (기본값 3)
        max_dist: 최대 거리 제한 (미터, 기본값 2000m)
    
    Returns:
        종합 지하철 점수 (0~1 범위)
    
    알고리즘:
    1. 모든 역과의 거리 계산
    2. max_dist 이내 역만 필터링
    3. 가장 가까운 top_n개 선택
    4. 각 역에 대해:
       - 거리 점수 계산 (subway_distance_score)
       - 역 기본 점수 가져오기 (Peak + Size)
       - 거리 가중치 계산: w = 1/(1 + d/250)
    5. 가중 평균 계산
    """
    # 역 데이터가 없으면 0점 반환
    if len(station_coords_arr) == 0:
        return 0.0
    
    # 매물에서 모든 역까지의 거리 계산
    dists = np.array([
        haversine(p_lat, p_lng, s_lat, s_lng)
        for (s_lat, s_lng) in station_coords_arr
    ])
    
    # max_dist 이내 역만 선택
    mask = dists <= max_dist
    if not mask.any():
        # 2km 이내에 역이 없으면 0점
        return 0.0
    
    # 유효한 역들의 거리와 점수
    d_valid = dists[mask]
    s_valid = station_scores_arr[mask]
    
    # 가장 가까운 top_n개 선택
    idx = np.argsort(d_valid)[:top_n]
    d_top = d_valid[idx]
    s_top = s_valid[idx]
    
    # 거리 점수 계산
    d_score = np.array([subway_distance_score(d) for d in d_top])
    
    # 거리 가중치 계산 (가까울수록 높은 가중치)
    # w = 1/(1 + d/250)
    # 예: d=0m → w=1.0, d=250m → w=0.5, d=500m → w=0.33
    w = 1 / (1 + d_top / 250.0)
    
    # 종합 점수 = (역 기본 점수 × 거리 점수)의 가중 평균
    combined = s_top * d_score
    return float(np.sum(combined * w) / np.sum(w))

# 매물별 지하철 점수 계산
if len(listings_valid) > 0:
    print("매물별 지하철 점수 계산 중...")
    sub_scores = []
    
    for idx, row in listings_valid.iterrows():
        # 진행 상황 출력 (500개마다)
        if idx % 500 == 0:
            print(f"  진행: {idx}/{len(listings_valid)}")
        
        # 지하철 점수 계산
        score = compute_subway_score_for_property(row["lat"], row["lng"])
        sub_scores.append(score)
    
    # DataFrame에 저장
    listings_valid["S_subway_raw"] = sub_scores
    
    print("\n지하철 점수 계산 완료!")
    print(listings_valid["S_subway_raw"].describe())
    # count: 매물 개수
    # mean: 평균 점수
    # std: 점수의 표준편차
    # min~max: 최소~최대 점수
else:
    print("유효한 매물이 없습니다. 지오코딩을 먼저 수행하세요.")


6. 지하철 점수 계산 함수
매물별 지하철 점수 계산 중...
  진행: 0/6

지하철 점수 계산 완료!
count    6.000000
mean     0.020079
std      0.018776
min      0.000000
25%      0.002525
50%      0.022558
75%      0.035016
max      0.040341
Name: S_subway_raw, dtype: float64


## 7. 버스 점수 계산

In [18]:
"""
===================================================================================
8. 버스 점수 계산
===================================================================================
목적:
1. 전국 버스정류장 데이터에서 서울 정류장만 필터링
2. 매물별로 버스 접근성 점수 계산

점수 체계:
- 거리 점수 (D_bus): exp(-d_min/300)
  → 가장 가까운 정류장까지의 거리로 계산
  → 300m = 도보 4분 기준
- 밀도 점수 (Density_bus): min(n_in_radius/8, 1.0)
  → 반경 300m 내 정류장 개수
  → 8개 이상이면 만점

최종 점수: S_bus = 0.7 × D_bus + 0.3 × Density_bus
===================================================================================
"""

print("\n" + "=" * 60)
print("7. 버스 점수 계산")
print("=" * 60)

# 버스 정류장 데이터 로드
bus = pd.read_csv(BUS_CSV, encoding="utf-8")
print(f"버스 정류장 데이터 shape: {bus.shape}")  # 전국 데이터

# 서울 버스정류장만 필터링
# "도시명" 컬럼에 "서울" 포함된 행만 선택
bus_seoul = bus[bus["도시명"].astype(str).str.contains("서울", na=False)].copy()
print(f"서울 버스정류장: {len(bus_seoul)}개")  # 약 16,980개

# 버스 정류장 좌표 배열 (속도 최적화)
bus_coords_arr = bus_seoul[["위도", "경도"]].to_numpy()

def compute_bus_score_for_property(p_lat, p_lng, radius=300):
    """
    매물 1건에 대한 버스 점수 계산
    
    Args:
        p_lat, p_lng: 매물의 위도, 경도
        radius: 밀도 계산 반경 (미터, 기본값 300m)
    
    Returns:
        버스 점수 (0~1 범위)
    
    계산 방법:
    1. 거리 점수 (D_bus):
       - 가장 가까운 정류장까지 거리 d_min
       - D_bus = exp(-d_min/300)
       - 예: 0m → 1.0, 150m → 0.61, 300m → 0.37, 600m → 0.14
    
    2. 밀도 점수 (Density_bus):
       - 반경 300m 내 정류장 개수 n_in_radius
       - Density_bus = min(n_in_radius/8, 1.0)
       - 예: 0개 → 0.0, 4개 → 0.5, 8개 → 1.0, 16개 → 1.0
    
    3. 종합 점수:
       - S_bus = 0.7 × D_bus + 0.3 × Density_bus
       - 거리가 더 중요 (70%), 밀도는 보조 (30%)
    """
    # 버스 정류장 데이터가 없으면 0점
    if len(bus_coords_arr) == 0:
        return 0.0
    
    # 매물에서 모든 버스 정류장까지의 거리 계산
    dists = np.array([
        haversine(p_lat, p_lng, b_lat, b_lng)
        for (b_lat, b_lng) in bus_coords_arr
    ])
    
    # 1) 최소 거리 (가장 가까운 정류장)
    d_min = dists.min()
    
    # 2) 반경 내 정류장 개수
    n_in_radius = (dists <= radius).sum()
    
    # 3) 거리 점수 계산
    # exp(-d_min/300): 거리가 300m 증가할 때마다 점수가 e배 감소
    D_bus = np.exp(-d_min / 300.0)
    
    # 4) 밀도 점수 계산
    # 8개 정류장 = 만점 (1.0)
    Density_bus = min(n_in_radius / 8.0, 1.0)
    
    # 5) 종합 점수: 거리 70%, 밀도 30%
    return float(0.7 * D_bus + 0.3 * Density_bus)

# 매물별 버스 점수 계산
if len(listings_valid) > 0:
    print("매물별 버스 점수 계산 중...")
    bus_scores = []
    
    for idx, row in listings_valid.iterrows():
        # 진행 상황 출력 (500개마다)
        if idx % 500 == 0:
            print(f"  진행: {idx}/{len(listings_valid)}")
        
        # 버스 점수 계산
        score = compute_bus_score_for_property(row["lat"], row["lng"])
        bus_scores.append(score)
    
    # DataFrame에 저장
    listings_valid["S_bus_raw"] = bus_scores
    
    print("\n버스 점수 계산 완료!")
    print(listings_valid["S_bus_raw"].describe())
    # count: 매물 개수
    # mean: 평균 0.63 (서울은 버스망이 잘 구축되어 있음)
    # std: 표준편차
    # min~max: 0.51 ~ 0.94 (모든 매물이 어느 정도 버스 접근성 있음)
else:
    print("유효한 매물이 없습니다.")


7. 버스 점수 계산
버스 정류장 데이터 shape: (259151, 9)
서울 버스정류장: 16980개
매물별 버스 점수 계산 중...
  진행: 0/6

버스 점수 계산 완료!
count    6.000000
mean     0.627758
std      0.167568
min      0.513014
25%      0.516917
50%      0.549046
75%      0.672787
max      0.935204
Name: S_bus_raw, dtype: float64


## 8. 최종 교통 온도 계산

In [19]:
"""
===================================================================================
9. 최종 교통온도 계산
===================================================================================
목적:
1. 지하철/버스 점수를 정규화하여 통합
2. 교통 점수를 30~43°C 스케일의 "온도"로 변환
3. 결과를 CSV로 저장하고 분석

교통온도 공식:
- S_transport = 0.55 × S_subway_norm + 0.45 × S_bus_norm
- TrafficTemp = 30 + 13 × S_transport (°C)

온도 해석:
- 30-34°C: 교통 불편 (역세권 밖, 버스 적음)
- 34-39°C: 평균~양호 (일반적인 서울 주거지)
- 39-43°C: 교통 우수 (역세권, 버스망 우수)
===================================================================================
"""

print("\n" + "=" * 60)
print("8. 최종 교통온도 계산")
print("=" * 60)

if len(listings_valid) > 0:
    # Min-Max 정규화 함수 (컬럼 단위)
    def minmax_norm_column(df, col):
        """
        DataFrame의 특정 컬럼을 0~1 범위로 정규화
        
        Args:
            df: DataFrame
            col: 정규화할 컬럼명
        
        Returns:
            정규화된 Series
        """
        s = df[col]
        s_min, s_max = s.min(), s.max()
        
        # 최대값과 최소값이 같으면 0 반환
        if s_max - s_min == 0:
            return pd.Series(0.0, index=df.index)
        
        # Min-Max 정규화 적용
        return (s - s_min) / (s_max - s_min)
    
    # 지하철/버스 점수 정규화 (0~1 범위)
    # 각 매물의 상대적 순위를 0~1로 변환
    listings_valid["S_subway_norm"] = minmax_norm_column(listings_valid, "S_subway_raw")
    listings_valid["S_bus_norm"] = minmax_norm_column(listings_valid, "S_bus_raw")
    
    # 지하철:버스 = 55:45 비율로 통합
    # 지하철이 약간 더 중요하게 고려됨 (장거리 이동에 유리)
    W_SUBWAY = 0.55
    W_BUS = 0.45
    
    # 통합 교통 점수 계산 (0~1 범위)
    listings_valid["S_transport"] = (
        W_SUBWAY * listings_valid["S_subway_norm"]    # 지하철 기여도 55%
        + W_BUS * listings_valid["S_bus_norm"]        # 버스 기여도 45%
    )
    
    # 교통온도 계산 (30~43°C 스케일)
    # TrafficTemp = 30 + 13 × S_transport
    #
    # 공식 설명:
    # - S_transport = 0.0 → 30°C (최저 온도, 교통 불편)
    # - S_transport = 0.5 → 36.5°C (평균 온도, 보통 수준)
    # - S_transport = 1.0 → 43°C (최고 온도, 교통 우수)
    #
    # 대안 공식: TrafficTemp = 36.5 + 13 × (S_transport - 0.5)
    # → 36.5°C를 중심으로 ±6.5°C 범위
    listings_valid["TrafficTemp"] = 30 + 13 * listings_valid["S_transport"]
    
    print("교통온도 계산 완료!")
    print("\n교통온도 통계:")
    print(listings_valid["TrafficTemp"].describe())
    # count: 매물 개수
    # mean: 평균 온도
    # std: 표준편차
    # min: 최저 온도 (교통이 가장 불편한 매물)
    # max: 최고 온도 (교통이 가장 좋은 매물)
    
    # 결과 확인 (샘플)
    print("\n샘플 결과:")
    result_cols = ["listing_id", "address", "property_type", "S_subway_norm", "S_bus_norm", "TrafficTemp"]
    print(listings_valid[result_cols].head(10))
    
    # 결과 저장
    output_file = BASE_DIR / "listings_with_traffic_temp.csv"
    listings_valid.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n결과를 '{output_file}'에 저장했습니다.")
    
    # 교통온도 분포 확인
    print("\n교통온도 구간별 분포:")
    bins = [30, 34, 39, 43]  # 구간 경계
    labels = ["낮음 (30-34°C)", "보통 (34-39°C)", "높음 (39-43°C)"]
    
    # 구간별 분류
    listings_valid["TrafficTemp_category"] = pd.cut(
        listings_valid["TrafficTemp"], 
        bins=bins,           # 구간 경계값
        labels=labels,       # 구간 레이블
        include_lowest=True  # 최소값 포함
    )
    
    # 구간별 매물 수 출력
    print(listings_valid["TrafficTemp_category"].value_counts().sort_index())
    
    # 해석:
    # - 낮음: 교통이 불편한 지역 (역세권 밖, 버스 적음)
    # - 보통: 일반적인 서울 주거지 (중간 수준 접근성)
    # - 높음: 교통이 우수한 지역 (역세권, 대중교통 편리)
    
else:
    print("유효한 매물이 없습니다. 지오코딩을 먼저 수행하세요.")


8. 최종 교통온도 계산
교통온도 계산 완료!

교통온도 통계:
count     6.000000
mean     35.148715
std       3.338069
min      30.216326
25%      33.391716
50%      36.028148
75%      36.206297
max      39.841082
Name: TrafficTemp, dtype: float64

샘플 결과:
  listing_id                        address property_type  S_subway_norm  \
0   18446002      서울특별시 서초구 양재동 116-3 소호M타워         원투룸_2       1.000000   
1   18350009     서울특별시 서초구 양재동 208-9 화평빌라타운         원투룸_2       0.000000   
2   18448077  서울특별시 서초구 양재동 257-8 다비치 스위트 홈         원투룸_2       0.868014   
3   18320532       서울특별시 서초구 양재동 342-8 리더스빌         원투룸_2       0.250359   
4   18306808            서울특별시 서초구 양재동 366-9         원투룸_2       0.000000   
5   18443369  서울특별시 서초구 양재동 257-8 다비치 스위트 홈         원투룸_2       0.868014   

   S_bus_norm  TrafficTemp  
0    0.460014    39.841082  
1    1.000000    35.850000  
2    0.000000    36.206297  
3    0.133713    32.572288  
4    0.036979    30.216326  
5    0.000000    36.206297  

결과를 'c:\Users\Playdata\OneDrive\

---

In [ ]:
"""
===================================================================================
10. 추가 분석 - 자치구별 및 매물 유형별 교통온도 통계
===================================================================================
목적:
1. 자치구별로 교통온도 평균, 표준편차 등 통계 계산
2. 매물 유형(원투룸, 오피스텔 등)별 교통온도 비교
3. 지역별/유형별 교통 편의성 패턴 파악

활용:
- 어느 자치구가 교통이 가장 편리한지 확인
- 매물 유형별 교통 접근성 차이 분석
===================================================================================
"""

print("\n" + "=" * 60)
print("9. 자치구별 교통온도 통계")
print("=" * 60)

if len(listings_valid) > 0:
    # 주소에서 자치구 추출 함수
    def extract_district(address):
        """
        주소 문자열에서 서울시 자치구 추출
        
        Args:
            address: 전체 주소 (예: "서울특별시 강남구 역삼동 123-4")
        
        Returns:
            자치구명 (예: "강남구") 또는 "기타"
        
        방법:
        - 공백으로 분리한 후 "구"로 끝나는 부분 찾기
        - 서울시 25개 자치구 중 하나를 반환
        """
        parts = address.split()
        for part in parts:
            if part.endswith("구"):
                return part
        return "기타"
    
    # 각 매물의 자치구 추출
    listings_valid["자치구"] = listings_valid["address"].apply(extract_district)
    
    # 자치구별 통계 계산
    # groupby: 자치구별로 그룹화
    # agg: 여러 집계 함수 동시 적용
    district_stats = listings_valid.groupby("자치구").agg({
        "TrafficTemp": ["count", "mean", "std", "min", "max"],  # 교통온도 통계
        "S_subway_norm": "mean",  # 평균 지하철 점수
        "S_bus_norm": "mean"      # 평균 버스 점수
    }).round(2)  # 소수점 2자리로 반올림
    
    # 컬럼명 정리 (다중 인덱스 → 단일 인덱스)
    district_stats.columns = ["매물수", "평균온도", "표준편차", "최소온도", "최대온도", "지하철점수", "버스점수"]
    
    # 평균온도 기준으로 내림차순 정렬
    district_stats = district_stats.sort_values("평균온도", ascending=False)
    
    print("\n자치구별 교통온도 통계:")
    print(district_stats)
    # 해석:
    # - 평균온도가 높은 자치구 = 대중교통이 편리한 지역
    # - 표준편차가 큰 자치구 = 매물 간 교통 편차가 큼
    # - 지하철점수 vs 버스점수 비교로 교통 특성 파악
    
    # 자치구별 통계를 CSV로 저장
    district_stats.to_csv(BASE_DIR / "traffic_temp_by_district.csv", encoding="utf-8-sig")
    print("\n자치구별 통계를 'traffic_temp_by_district.csv'에 저장했습니다.")
    
    # 매물 유형별 통계
    print("\n" + "=" * 60)
    print("매물 유형별 교통온도 통계")
    print("=" * 60)
    
    # 매물 유형별로 그룹화하여 통계 계산
    type_stats = listings_valid.groupby("property_type").agg({
        "TrafficTemp": ["count", "mean", "std", "min", "max"]
    }).round(2)
    
    # 컬럼명 정리
    type_stats.columns = ["매물수", "평균온도", "표준편차", "최소온도", "최대온도"]
    
    # 평균온도 기준으로 내림차순 정렬
    type_stats = type_stats.sort_values("평균온도", ascending=False)
    
    print(type_stats)
    # 해석:
    # - 오피스텔/원투룸 = 주로 역세권에 위치 (평균온도 높음)
    # - 아파트/빌라 = 주거지역에 분산 (평균온도 중간)
    # - 상가 = 주요 상권 중심 (평균온도 높거나 중간)
    
else:
    print("유효한 매물이 없습니다.")